# 1. What you'll learn

        - encode short documents as vectors with a small inference-only model or offline fallback
- implement cosine exact search and random-hyperplane LSH
- measure recall and candidate reduction instead of claiming approximate search is free

        **The one question this notebook answers:** How does semantic retrieval become nearest-neighbour search, and what does approximation buy and lose?

# 2. Setup

The notebook first tries the cached all-MiniLM-L6-v2 encoder in inference-only mode. If unavailable it prints a banner and uses TF-IDF, keeping the retrieval and indexing lesson runnable without a network.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A bundled corpus contains short paraphrases across food, travel, software, medicine, finance, sports, and science. Each document has a topic label used only to evaluate retrieval relevance.

**Small example:** For five one-dimensional similarities [0.12, 0.91, 0.44, 0.88, -0.1], exact top-2 returns indices 1 and 3. An approximate bucket that contains [0,1,4] checks only three items but misses index 3, giving recall@2 of 1/2.

In [ ]:
base={"food":["bake bread in a hot oven","make a loaf with flour and yeast","slice fresh sourdough for breakfast","knead dough until elastic","toast bread with butter"],"travel":["reserve a train ticket to Paris","book rail travel for France","find a cheap flight overseas","pack luggage for a city break","check the departure platform"],"software":["fix a Python memory leak","debug code that uses too much RAM","deploy a web service safely","write tests before refactoring","review an API error log"],"medicine":["measure blood pressure at home","monitor hypertension with a cuff","schedule a routine vaccination","ask a pharmacist about dosage","record resting heart rate"],"finance":["compare mortgage interest rates","estimate the monthly home loan payment","rebalance a retirement portfolio","read a company income statement","track household spending"],"sports":["practice a tennis backhand","train for a long distance race","watch the football final","improve a basketball free throw","recover after a cycling workout"],"science":["observe stars with a telescope","measure acidity in a chemistry lab","study how plants use sunlight","simulate planetary motion","record temperature during an experiment"]}
docs=[];labels=[]
for topic,items in base.items():
    for repeat in range(6):
        for text_item in items: docs.append(text_item+f" example {repeat}");labels.append(topic)
queries=["how do I cook a loaf?","diagnose excessive RAM use","buy rail passage to the French capital","check high blood pressure"]
query_topics=["food","software","travel","medicine"]
print("Documents:",len(docs),"queries:",len(queries),"topic counts:",pd.Series(labels).value_counts().to_dict())

# 4. Explore

We inspect topic counts, query-document token overlap, and the pairwise cosine matrix. These reveal why lexical retrieval struggles with paraphrases and why vector normalization matters.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
tokens=lambda s:set(w.strip("?.,").lower() for w in s.split())
overlap=[len(tokens(queries[0])&tokens(d)) for d in docs]
tf=TfidfVectorizer(stop_words="english").fit_transform(docs)
small=(tf[:25]@tf[:25].T).toarray()
fig,axes=plt.subplots(1,3,figsize=(14,3.7));pd.Series(labels).value_counts().plot.bar(ax=axes[0],color="#176b66");axes[0].set_title("Balanced evaluation topics")
axes[1].hist(overlap,bins=np.arange(-.5,4.5,1),color="#d38b45");axes[1].set(title="Literal overlap for a paraphrase",xlabel="shared tokens")
im=axes[2].imshow(small,cmap="mako",vmin=0,vmax=1);axes[2].set_title("TF-IDF cosine, first 25 docs");fig.colorbar(im,ax=axes[2],fraction=.046);plt.tight_layout();plt.show()

**Takeaway:** Equal topic counts make recall comparisons interpretable.

**Takeaway:** Most relevant paraphrases share no literal query token, motivating learned semantic encoders.

**Takeaway:** Within-topic cosine blocks exist, but repeated template words also create misleading similarity.

# 5. Prepare

Encoding is inference only. We normalize every vector so an inner product equals cosine similarity. The fallback fits vocabulary only on documents; queries never alter the representation.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
encoder_name="TF-IDF fallback"
try:
    from sentence_transformers import SentenceTransformer
    encoder=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",local_files_only=True)
    doc_vec=encoder.encode(docs,normalize_embeddings=True,show_progress_bar=False);query_vec=encoder.encode(queries,normalize_embeddings=True,show_progress_bar=False);encoder_name="cached all-MiniLM-L6-v2 (inference only)"
except Exception as exc:
    print("OFFLINE BANNER: pretrained MiniLM was not cached; using deterministic TF-IDF/SVD fallback.")
    vectorizer=TfidfVectorizer(stop_words="english",ngram_range=(1,2));all_doc=vectorizer.fit_transform(docs);all_query=vectorizer.transform(queries)
    doc_vec=normalize(all_doc).toarray();query_vec=normalize(all_query).toarray()
print("Encoder:",encoder_name,"document matrix:",doc_vec.shape,"query matrix:",query_vec.shape)

# 6. Train

        Exact search scans every normalized document. Our approximate index hashes vectors by the signs of random projections, then searches the exact cosine scores only inside buckets with small Hamming distance.

        Conceptually, fitting follows these steps:

        1. Normalize document vectors.
2. Project each vector onto seeded random hyperplanes.
3. Turn projection signs into a compact binary bucket key.
4. At query time search matching or nearby buckets, then rerank candidates exactly.

In [ ]:
rng=np.random.default_rng(SEED);n_bits=7;planes=rng.normal(size=(n_bits,doc_vec.shape[1]))
def bits(V): return (V@planes.T>=0).astype(np.uint8)
doc_bits=bits(doc_vec);query_bits=bits(query_vec)
def exact_top(q,k=5): return np.argsort(-(doc_vec@q))[:k]
def approx_top(q,qb,k=5,radius=1):
    ham=np.sum(doc_bits!=qb,axis=1);cand=np.flatnonzero(ham<=radius)
    if len(cand)<k: cand=np.argsort(ham)[:max(k,20)]
    return cand[np.argsort(-(doc_vec[cand]@q))[:k]],len(cand)
for q,topic,v,b in zip(queries,query_topics,query_vec,query_bits):
    ids,c=approx_top(v,b);print(f"{q!r} ->",[(labels[i],docs[i][:32]) for i in ids[:3]],"candidates",c)

# 7. Evaluate

We report topic precision, exact top-5 overlap, and the share of corpus scored. Timing is repeated on batched synthetic queries because one tiny query is below reliable clock resolution.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
exact=[exact_top(q) for q in query_vec];approx=[approx_top(q,b) for q,b in zip(query_vec,query_bits)]
topic_precision=np.mean([np.mean([labels[i]==t for i in ids]) for t,ids in zip(query_topics,exact)])
recall=np.mean([len(set(e)&set(a))/5 for e,(a,c) in zip(exact,approx)]);candidate_share=np.mean([c/len(docs) for a,c in approx])
repeats=500;t0=time.perf_counter();[exact_top(query_vec[i%len(queries)]) for i in range(repeats)];exact_ms=(time.perf_counter()-t0)*1000/repeats
t0=time.perf_counter();[approx_top(query_vec[i%len(queries)],query_bits[i%len(queries)]) for i in range(repeats)];approx_ms=(time.perf_counter()-t0)*1000/repeats
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(["topic precision@5","approx recall@5"],[topic_precision,recall],color=["#176b66","#d38b45"]);axes[0].set_ylim(0,1);axes[0].set_title("Retrieval quality")
axes[1].bar(["exact corpus","approx candidates"],[1,candidate_share],color=[".6","#176b66"]);axes[1].set_ylim(0,1);axes[1].set_title("Fraction scored")
axes[2].bar(["exact","approx"],[exact_ms,approx_ms],color=[".6","#d38b45"]);axes[2].set(title="Python demo latency",ylabel="milliseconds/query");fig.suptitle(encoder_name);plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Exact topic precision={topic_precision:.3f}; approximate recall={recall:.3f}; candidates={candidate_share:.1%}")

**Takeaway:** Approximation scores only a fraction of documents but can miss true neighbours; on this tiny corpus Python indexing overhead may make it slower, which is why production speed claims require realistic scale.

# 8. Break it

We intentionally skip normalization, then multiply one document vector by 100. Inner product now rewards magnitude rather than angle, and the inflated document can appear for unrelated queries.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
bad_docs=doc_vec.copy();bad_docs[0]*=100
bad_hits=[int(np.argmax(bad_docs@q)) for q in query_vec];good_hits=[int(exact_top(q,1)[0]) for q in query_vec]
print("Top hits with cosine normalization:",[labels[i] for i in good_hits]);print("Top hits after one magnitude explosion:",[labels[i] for i in bad_hits]);print("Unrelated queries hijacked by doc 0:",bad_hits.count(0))

**Use this when…** you need to retrieve semantically or geometrically similar items from a corpus and can define an embedding whose neighbourhoods match relevance.

        **Don't use this when…** exact keywords, filters, freshness, or calibrated relevance matter more than embedding similarity, or the evaluation set does not represent user queries.

        ## Try this

        1. Change hash bits from 4 to 12; expect smaller buckets and potentially lower recall.
2. Search Hamming radius 0, 1, and 2; chart candidates against recall.
3. Cache MiniLM with data/download.py and compare paraphrases that share no words.